In [1]:
import os
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# ==============================================================================
# BƯỚC 1: NẠP DỮ LIỆU TẬP TRAIN (80%) TỪ datasets/processed & CHUẨN HÓA [0, 1]
# ==============================================================================
candidate_roots = [
    os.path.abspath(os.path.join(os.getcwd(), '..')),
    os.path.abspath(os.getcwd()),
    os.path.abspath(os.path.join(os.getcwd(), '..', 'FMN_EduWarning')),
    '/content/drive/MyDrive/FMNN_CODE/FMN_EduWarning'
]
project_root = next((r for r in candidate_roots if os.path.exists(os.path.join(r, 'datasets', 'processed'))), candidate_roots[0])
processed_dir = os.path.join(project_root, 'datasets', 'processed')
results_dir = os.path.join(project_root, 'results')
os.makedirs(results_dir, exist_ok=True)

parquet_path = os.path.join(processed_dir, 'cleaned_data.parquet')
json_path = os.path.join(processed_dir, 'cleaned_data.json')
df_all = pd.read_parquet(parquet_path) if os.path.exists(parquet_path) else pd.read_json(json_path, orient='records')

FEATURE_COLS = ['TBC HK', 'TBC TL', 'Số TC Đạt', 'Số TC F', 'Kỳ xét']
df_train = df_all[df_all['split'] == 'train'].reset_index(drop=True)
X_train_full = df_train[FEATURE_COLS].values.astype(float)
y_cb_train_full = df_train['Cảnh báo'].values.astype(int)

# Chuẩn hóa đặc trưng về [0, 1]
x_min = np.array([0.0, 0.0, 0.0, 0.0, 1.0])
x_max = np.array([
    4.0,
    4.0,
    max(141.0, float(np.max(X_train_full[:, 2]))),
    max(24.0, float(np.max(X_train_full[:, 3]))),
    max(7.0, float(np.max(X_train_full[:, 4])))
])
X_train_norm = np.clip((X_train_full - x_min) / (x_max - x_min + 1e-12), 0.0, 1.0)

In [2]:
# ==============================================================================
# BƯỚC 2: HÀM ĐÁNH GIÁ TỪNG FOLD VỚI CẶP THAM SỐ (THETA, GAMMA)
# ==============================================================================
def evaluate_fold(X_tr, y_tr, X_va, theta, gamma, eps=1e-12):
    n_features = X_tr.shape[1]
    inv_2n = 1.0 / (2.0 * n_features)
    theta_map = {0: theta * 0.8, 1: theta * 1.2}
    V = np.empty((2048, n_features), dtype=float)
    W = np.empty((2048, n_features), dtype=float)
    L = np.empty(2048, dtype=y_tr.dtype)
    n_boxes = 0

    for x, label in zip(X_tr, y_tr):
        if n_boxes == 0:
            V[0], W[0], L[0] = x, x, label
            n_boxes = 1
            continue

        same_idx = np.flatnonzero(L[:n_boxes] == label)
        exp_id = -1
        bounds_changed = True
        limit = n_features * theta_map.get(int(label), theta) + eps

        if len(same_idx) > 0:
            Vs, Ws = V[same_idx], W[same_idx]
            inside = np.flatnonzero(np.all((x >= Vs) & (x <= Ws), axis=1))
            if len(inside) > 0:
                cand = int(same_idx[inside[0]])
                if np.sum(W[cand] - V[cand]) <= limit:
                    exp_id = cand
                    bounds_changed = False
            if exp_id == -1:
                mems = np.sum((1.0 - np.clip(gamma * (x - Ws), 0.0, 1.0)) + (1.0 - np.clip(gamma * (Vs - x), 0.0, 1.0)), axis=1) * inv_2n
                order = same_idx[np.argsort(-mems, kind='stable')]
                valid = np.flatnonzero(np.sum(np.maximum(W[order], x) - np.minimum(V[order], x), axis=1) <= limit)
                if len(valid) > 0:
                    exp_id = int(order[valid[0]])
                    V[exp_id] = np.minimum(V[exp_id], x)
                    W[exp_id] = np.maximum(W[exp_id], x)

        if exp_id == -1:
            exp_id = n_boxes
            if exp_id >= len(V):
                V = np.vstack([V, np.empty_like(V)])
                W = np.vstack([W, np.empty_like(W)])
                L = np.concatenate([L, np.empty_like(L)])
            V[exp_id], W[exp_id], L[exp_id] = x, x, label
            n_boxes += 1

        if bounds_changed:
            diff_idx = np.flatnonzero(L[:n_boxes] != label)
            if len(diff_idx) > 0:
                cands = diff_idx[np.all(W[exp_id] >= V[diff_idx] - eps, axis=1) & np.all(W[diff_idx] >= V[exp_id] - eps, axis=1)]
                for o_id in cands:
                    ov = []
                    for i in range(n_features):
                        vj, wj, vk, wk = V[exp_id, i], W[exp_id, i], V[o_id, i], W[o_id, i]
                        if wj < vk - eps or wk < vj - eps or (abs(wj - vk) <= eps and vj < vk - eps) or (abs(wk - vj) <= eps and vk < vj - eps):
                            ov = []
                            break
                        if vj < vk - eps and vk <= wj + eps and wj < wk - eps:
                            ov.append((wj - vk, i, 1))
                        elif vk < vj - eps and vj <= wk + eps and wk < wj - eps:
                            ov.append((wk - vj, i, 2))
                        elif vj <= vk + eps and wk <= wj + eps and (wj - vj) >= (wk - vk) - eps:
                            ov.append((min(wk - vj, wj - vk), i, 3))
                        elif vk <= vj + eps and wj <= wk + eps:
                            ov.append((min(wj - vk, wk - vj), i, 4))
                        else:
                            ov = []
                            break
                    if ov:
                        pos = [c for c in ov if c[0] > eps]
                        target = pos if pos else ov
                        target.sort(key=lambda t: t[0])
                        _, d, case = target[0]
                        vj, wj, vk, wk = V[exp_id, d], W[exp_id, d], V[o_id, d], W[o_id, d]
                        if case == 1:
                            V[o_id, d] = W[exp_id, d] = (wj + vk) / 2.0
                        elif case == 2:
                            V[exp_id, d] = W[o_id, d] = (wk + vj) / 2.0
                        elif case == 3:
                            if abs(vj - vk) <= eps and abs(wj - wk) <= eps:
                                W[exp_id, d] = V[o_id, d] = (vj + wj) / 2.0
                            elif (wk - vj) < (wj - vk):
                                V[exp_id, d] = wk
                            else:
                                W[exp_id, d] = vk
                        elif case == 4:
                            if abs(vj - vk) <= eps and abs(wj - wk) <= eps:
                                W[o_id, d] = V[exp_id, d] = (vj + wj) / 2.0
                            elif (wj - vk) < (wk - vj):
                                V[o_id, d] = wj
                            else:
                                W[o_id, d] = vj

    Vb, Wb, Lb = V[:n_boxes], W[:n_boxes], L[:n_boxes]
    Cb = (Vb + Wb) / 2.0
    mems = np.sum((1.0 - np.clip(gamma * (X_va[:, None, :] - Wb[None, :, :]), 0.0, 1.0)) + (1.0 - np.clip(gamma * (Vb[None, :, :] - X_va[:, None, :]), 0.0, 1.0)), axis=2) * inv_2n
    is_cand = np.abs(mems - np.max(mems, axis=1, keepdims=True)) <= eps
    preds = np.empty(len(X_va), dtype=Lb.dtype)
    for i in range(len(X_va)):
        c = np.flatnonzero(is_cand[i])
        preds[i] = Lb[c[0]] if len(c) == 1 else Lb[c[np.argmin(np.linalg.norm(Cb[c] - X_va[i], axis=1))]]
    return preds, n_boxes

In [3]:
# ==============================================================================
# 👉 BƯỚC 3: ĐIỀU CHỈNH THETA & GAMMA TẠI ĐÂY -> CHẠY 10-FOLD & LƯU KẾT QUẢ
# ==============================================================================
THETA = 0.12   # Kích thước tối đa của siêu hộp mờ (0 < THETA <= 1, ví dụ: 0.05, 0.10, 0.12, 0.15, 0.20, 0.30)
GAMMA = 1.0    # Độ nhạy độ dốc hàm thuộc mờ (GAMMA > 0, ví dụ: 0.5, 1.0, 2.0, 4.0)
# ==============================================================================

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
fold_records = []
report_lines = [
    "=" * 100,
    f"  ĐÁNH GIÁ 10-FOLD STRATIFIED CROSS-VALIDATION TRÊN TẬP TRAIN | THETA = {THETA} | GAMMA = {GAMMA}",
    "=" * 100,
    f"{'Fold':<8} | {'Accuracy':>9} | {'Precision':>9} | {'Recall':>9} | {'F1-Score':>9} | {'Sensitivity':>11} | {'Specificity':>11} | {'Boxes':>5} | {'Time':>5}",
    "-" * 100
]
for line in report_lines:
    print(line)

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_train_norm, y_cb_train_full), start=1):
    t0 = time.time()
    y_pred, n_boxes = evaluate_fold(
        X_train_norm[train_idx], y_cb_train_full[train_idx],
        X_train_norm[val_idx], THETA, GAMMA
    )
    elapsed = time.time() - t0
    y_va = y_cb_train_full[val_idx]

    acc = accuracy_score(y_va, y_pred) * 100.0
    prec, rec, f1, _ = precision_recall_fscore_support(y_va, y_pred, average='macro', zero_division=0)
    prec, rec, f1 = prec * 100.0, rec * 100.0, f1 * 100.0
    tn, fp, fn, tp = confusion_matrix(y_va, y_pred, labels=[0, 1]).ravel()
    sensitivity = (tp / (tp + fn) * 100.0) if (tp + fn) > 0 else 0.0
    specificity = (tn / (tn + fp) * 100.0) if (tn + fp) > 0 else 0.0

    fold_records.append({
        'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1-Score': f1,
        'Sensitivity': sensitivity, 'Specificity': specificity, 'Boxes': n_boxes, 'Time': elapsed
    })
    row_str = f"Fold {fold_idx:<3} | {acc:>8.2f}% | {prec:>8.2f}% | {rec:>8.2f}% | {f1:>8.2f}% | {sensitivity:>10.2f}% | {specificity:>10.2f}% | {n_boxes:>5d} | {elapsed:>4.2f}s"
    print(row_str)
    report_lines.append(row_str)

df_folds = pd.DataFrame(fold_records)
mean_vals = df_folds.mean()
std_vals = df_folds.std()

footer_lines = [
    "-" * 100,
    f"{'TB 10-F':<8} | {mean_vals['Accuracy']:>8.2f}% | {mean_vals['Precision']:>8.2f}% | {mean_vals['Recall']:>8.2f}% | {mean_vals['F1-Score']:>8.2f}% | {mean_vals['Sensitivity']:>10.2f}% | {mean_vals['Specificity']:>10.2f}% | {mean_vals['Boxes']:>5.1f} | {mean_vals['Time']:>4.2f}s",
    f"{'± Std':<8} | {std_vals['Accuracy']:>8.2f}% | {std_vals['Precision']:>8.2f}% | {std_vals['Recall']:>8.2f}% | {std_vals['F1-Score']:>8.2f}% | {std_vals['Sensitivity']:>10.2f}% | {std_vals['Specificity']:>10.2f}% | {std_vals['Boxes']:>5.1f} | {std_vals['Time']:>4.2f}s",
    "=" * 100
]
for line in footer_lines:
    print(line)
    report_lines.append(line)

# Lưu duy nhất 1 file bảng kết quả vào thư mục results
result_file_path = os.path.join(results_dir, f"result_theta_{THETA}_gamma_{GAMMA}.txt")
with open(result_file_path, 'w', encoding='utf-8') as f_out:
    f_out.write("\n".join(report_lines) + "\n")

  ĐÁNH GIÁ 10-FOLD STRATIFIED CROSS-VALIDATION TRÊN TẬP TRAIN | THETA = 0.12 | GAMMA = 1.0
Fold     |  Accuracy | Precision |    Recall |  F1-Score | Sensitivity | Specificity | Boxes |  Time
----------------------------------------------------------------------------------------------------
Fold 1   |    99.65% |    93.54% |    97.09% |    95.24% |      94.44% |      99.75% |   477 | 3.05s
Fold 2   |    99.65% |    92.66% |    98.46% |    95.37% |      97.22% |      99.69% |   479 | 3.06s
Fold 3   |    99.70% |    95.87% |    95.87% |    95.87% |      91.89% |      99.85% |   473 | 3.14s
Fold 4   |    99.75% |    96.00% |    97.22% |    96.60% |      94.59% |      99.85% |   478 | 3.53s
Fold 5   |    99.85% |    97.34% |    98.60% |    97.96% |      97.30% |      99.90% |   473 | 3.13s
Fold 6   |    99.45% |    91.15% |    94.42% |    92.72% |      89.19% |      99.64% |   468 | 2.91s
Fold 7   |    99.75% |    94.97% |    98.55% |    96.69% |      97.30% |      99.80% |   484 | 3.22s


---
### 📖 Giải thích quy trình 10-Fold Stratified Cross-Validation và cách chọn $\theta$ (`THETA`), $\gamma$ (`GAMMA`) tốt nhất

#### 1. Tại sao dùng `StratifiedKFold` (10-Fold) trên tập `train`?
- Tập dữ liệu `train` (`19,970` sinh viên) có sự mất cân bằng lớn giữa nhóm **Bình thường (`98.16%`)** và nhóm **Cảnh báo học tập (`1.84%`)**.
- `StratifiedKFold(n_splits=10)` chia tập `train` thành **10 phần (10 Folds) bằng nhau**, đồng thời **giữ nguyên tỷ lệ phần trăm sinh viên bị cảnh báo học tập ở cả 10 Fold**.
- Trong mỗi lần thử một cặp tham số `(THETA, GAMMA)`, dữ liệu được đánh giá qua 10 vòng: mỗi vòng lấy **9 Fold huấn luyện** và **1 Fold kiểm thử**, sau đó lấy **trung bình cộng (`TB 10-F`)** của 10 Fold để chọn ra bộ `(THETA, GAMMA)` tối ưu trước khi đưa sang huấn luyện mô hình chính ở `FMN_EduWarning.ipynb`.

#### 2. Ý nghĩa khi điều chỉnh 2 tham số `THETA` ($\theta$) và `GAMMA` ($\gamma$)
- **`THETA` ($\theta \in (0, 1]$) – Kích thước tối đa của siêu hộp mờ**:
  - **Khi giảm `THETA` (ví dụ `0.05` – `0.12`)**: Các siêu hộp mờ nhỏ lại, bám sát ranh giới giữa sinh viên bình thường và sinh viên bị cảnh báo $\rightarrow$ **Tăng độ chính xác (`F1-Score`, `Sensitivity`)**, nhưng số lượng hộp mờ (`Boxes`) sinh ra nhiều hơn.
  - **Khi tăng `THETA` (ví dụ `0.25` – `0.50`)**: Các siêu hộp mờ phình to, gom nhiều sinh viên vào chung một hộp $\rightarrow$ Số lượng hộp mờ (`Boxes`) giảm mạnh, nhưng dễ bị chồng lấn vùng biên làm giảm `Sensitivity` và `F1-Score`.
- **`GAMMA` ($\gamma > 0$) – Độ nhạy của hàm thuộc mờ**:
  - Điều khiển độ dốc khi điểm dữ liệu nằm ngoài biên của siêu hộp mờ (giá trị thường dùng: `0.5`, `1.0`, `2.0`, `4.0`).

#### 3. Các chỉ số đo lường trên bảng kết quả để chọn bộ `(THETA, GAMMA)` tốt nhất
- **`F1-Score` *(Chỉ số quan trọng nhất)***: Trung bình điều hòa giữa `Precision` và `Recall`, phản ánh chính xác nhất chất lượng bộ tham số trên dữ liệu mất cân bằng.
- **`Sensitivity` *(Độ nhạy)*:** Tỷ lệ phát hiện đúng sinh viên thực sự bị cảnh báo học tập ($\frac{TP}{TP + FN}$). Chỉ số này càng cao thì càng ít bỏ sót sinh viên có nguy cơ.
- **`Specificity` *(Độ đặc hiệu)*:** Tỷ lệ nhận diện đúng sinh viên học tập bình thường ($\frac{TN}{TN + FP}$).
- **`Precision` & `Accuracy`**: Độ chuẩn xác khi dự báo cảnh báo và độ chính xác tổng thể.